# Laboratoria 07 - protokół z wykorzystaniem schematów postkwantowych

Twoim zadaniem jest wykorzystanie wybranych elementów spośród poniższych prymitywów do konstrukcji postkwantowego protokołu wymiany klucza sesyjnego opartego na poniższym protokole Fujioki:

| Initiator: $(ssk_i,spk_i)\leftarrow \mathsf{KEM2.Gen}()$ | | Responder: $(ssk_r,spk_r)\leftarrow \mathsf{KEM2.Gen}()$ |
|----------| -------- |----------|
| $$(esk_i,epk_i)\leftarrow \mathsf{KEM1.Gen}()$$ | | |
| $$r_1 \leftarrow\{0,1\}^{\lambda},\;(c_1,k_1)\leftarrow \mathsf{KEM2.Enc}(spk_r,r_1)$$ | | |
| |$$\xrightarrow{\qquad epk_i,\;c_1 \qquad}$$ | |
| | |$$r_2 \xleftarrow{\ }\{0,1\}^{\lambda},\;(c_2,k_2)\leftarrow \mathsf{KEM2.Enc}(spk_i,r_2)$$ |
| | |$$r_3 \xleftarrow{\ }\{0,1\}^{\lambda},\;(c_3,k_3)\leftarrow \mathsf{KEM1.Enc}(epk_i,r_3)$$ |
| |$$\xleftarrow{\qquad c_2,\;c_3 \qquad}$$ | |
| | |$$k_1\leftarrow\mathsf{KEM2.Dec}(ssk_r,c_1)$$|

## K-PKE

In [7]:
# Uwaga: to jest demonstracja dydaktyczna, NIE bezpieczna kryptograficznie.

from dataclasses import dataclass
from hashlib import sha3_512, shake_128, shake_256
from secrets import token_bytes
from typing import List, Tuple, Optional


# ============================================================
# Parametry
# ============================================================

N = 256
Q = 3329

Wielomian = List[int]
Wektor = List[Wielomian]
Macierz = List[List[Wielomian]]

KluczPubliczny = Tuple[bytes, Wektor]   # (rho, t)
KluczPrywatny = Wektor                 # s
Szyfrogram = Tuple[Wektor, Wielomian]  # (u, v)


@dataclass(frozen=True)
class ParametryKPKE:
    nazwa: str
    k: int
    eta1: int
    eta2: int


PARAMETRY = {
    "ML-KEM-512": ParametryKPKE("ML-KEM-512", k=2, eta1=3, eta2=2),
    "ML-KEM-768": ParametryKPKE("ML-KEM-768", k=3, eta1=2, eta2=2),
    "ML-KEM-1024": ParametryKPKE("ML-KEM-1024", k=4, eta1=2, eta2=2),
}


# ============================================================
# Arytmetyka w R_q = Z_q[X] / (X^N + 1)
# ============================================================

class Rq:
    """Arytmetyka wielomianów w R_q = Z_q[X] / (X^N + 1)."""

    @staticmethod
    def zero() -> Wielomian:
        return [0] * N

    @staticmethod
    def redukuj(a: Wielomian) -> Wielomian:
        return [x % Q for x in a]

    @staticmethod
    def dodaj(a: Wielomian, b: Wielomian) -> Wielomian:
        return [(x + y) % Q for x, y in zip(a, b)]

    @staticmethod
    def odejmij(a: Wielomian, b: Wielomian) -> Wielomian:
        return [(x - y) % Q for x, y in zip(a, b)]

    @staticmethod
    def pomnoz(a: Wielomian, b: Wielomian) -> Wielomian:
        """Naiwne mnożenie O(N^2) z redukcją przez X^N + 1."""
        wynik = [0] * N

        for i, ai in enumerate(a):
            for j, bj in enumerate(b):
                stopien = i + j
                iloczyn = ai * bj

                if stopien < N:
                    wynik[stopien] += iloczyn
                else:
                    wynik[stopien - N] -= iloczyn

        return [x % Q for x in wynik]

    @staticmethod
    def iloczyn_skalarny(a: Wektor, b: Wektor) -> Wielomian:
        if len(a) != len(b):
            raise ValueError("Wektory muszą mieć tę samą długość.")

        wynik = Rq.zero()
        for ai, bi in zip(a, b):
            wynik = Rq.dodaj(wynik, Rq.pomnoz(ai, bi))
        return wynik

    @staticmethod
    def macierz_razy_wektor(A: Macierz, x: Wektor) -> Wektor:
        if any(len(wiersz) != len(x) for wiersz in A):
            raise ValueError("Niezgodne rozmiary macierzy i wektora.")
        return [Rq.iloczyn_skalarny(wiersz, x) for wiersz in A]

    @staticmethod
    def transpozycja_razy_wektor(A: Macierz, x: Wektor) -> Wektor:
        """Oblicza A^T * x bez jawnego budowania transpozycji."""
        k = len(A)
        if len(x) != k or any(len(wiersz) != k for wiersz in A):
            raise ValueError("Macierz A musi być kwadratowa k x k.")

        wynik = []
        for kolumna in range(k):
            suma = Rq.zero()
            for wiersz in range(k):
                skladnik = Rq.pomnoz(A[wiersz][kolumna], x[wiersz])
                suma = Rq.dodaj(suma, skladnik)
            wynik.append(suma)
        return wynik


# ============================================================
# Generowanie macierzy i szumu
# ============================================================

def _losowy_wielomian_jednorodny(rho: bytes, i: int, j: int) -> Wielomian:
    """Generuje wielomian współczynnikowy przez SHAKE128 i rejection sampling."""
    if not (0 <= i <= 255 and 0 <= j <= 255):
        raise ValueError("Indeksy macierzy muszą mieścić się w bajcie.")

    wejscie = rho + bytes([j, i])
    wynik = []
    dlugosc = 768

    while len(wynik) < N:
        bufor = shake_128(wejscie).digest(dlugosc)
        wynik.clear()

        for pozycja in range(0, len(bufor) - 2, 3):
            b0 = bufor[pozycja]
            b1 = bufor[pozycja + 1]
            b2 = bufor[pozycja + 2]

            d1 = b0 | ((b1 & 0x0F) << 8)
            d2 = (b1 >> 4) | (b2 << 4)

            if d1 < Q:
                wynik.append(d1)
                if len(wynik) == N:
                    break

            if d2 < Q:
                wynik.append(d2)
                if len(wynik) == N:
                    break

        if len(wynik) < N:
            dlugosc *= 2

    return wynik


def _generuj_macierz(rho: bytes, k: int) -> Macierz:
    return [
        [_losowy_wielomian_jednorodny(rho, i, j) for j in range(k)]
        for i in range(k)
    ]


def _cbd(bufor: bytes, eta: int) -> Wielomian:
    """Próbkowanie z rozkładu centered binomial distribution CBD_eta."""
    potrzebne_bity = N * 2 * eta
    if len(bufor) * 8 < potrzebne_bity:
        raise ValueError("Bufor jest zbyt krótki dla CBD.")

    def bit(numer: int) -> int:
        return (bufor[numer // 8] >> (numer % 8)) & 1

    wynik = []
    for i in range(N):
        poczatek = 2 * eta * i
        a = sum(bit(poczatek + j) for j in range(eta))
        b = sum(bit(poczatek + eta + j) for j in range(eta))
        wynik.append((a - b) % Q)
    return wynik


def _prf_cbd(seed: bytes, nonce: int, eta: int) -> Wielomian:
    if len(seed) != 32:
        raise ValueError("Seed musi mieć dokładnie 32 bajty.")
    if not 0 <= nonce <= 255:
        raise ValueError("Nonce musi mieścić się w jednym bajcie.")

    bufor = shake_256(seed + bytes([nonce])).digest(64 * eta)
    return _cbd(bufor, eta)


def _wektor_cbd(seed: bytes, poczatkowy_nonce: int, liczba: int, eta: int) -> Wektor:
    return [
        _prf_cbd(seed, poczatkowy_nonce + i, eta)
        for i in range(liczba)
    ]


# ============================================================
# Kodowanie 32-bajtowej wiadomości
# ============================================================

def _wiadomosc_na_wielomian(wiadomosc: bytes) -> Wielomian:
    if len(wiadomosc) != 32:
        raise ValueError("K-PKE szyfruje dokładnie 32-bajtową wiadomość.")

    polowa_q = (Q + 1) // 2
    wynik = []
    for bajt in wiadomosc:
        for pozycja in range(8):
            bit = (bajt >> pozycja) & 1
            wynik.append(polowa_q if bit else 0)
    return wynik


def _wielomian_na_wiadomosc(a: Wielomian) -> bytes:
    if len(a) != N:
        raise ValueError(f"Wielomian musi mieć {N} współczynników.")

    bity = []
    for wspolczynnik in a:
        x = wspolczynnik % Q
        bit = ((2 * x + Q // 2) // Q) & 1
        bity.append(bit)

    wynik = bytearray(32)
    for i, bit in enumerate(bity):
        wynik[i // 8] |= bit << (i % 8)
    return bytes(wynik)


# ============================================================
# K-PKE
# ============================================================

def KeyGenK_PKE(
    zestaw: str = "ML-KEM-768",
    seed: Optional[bytes] = None,
) -> Tuple[KluczPubliczny, KluczPrywatny]:
    """Generuje (klucz_publiczny, klucz_prywatny), gdzie t = A*s + e."""
    if zestaw not in PARAMETRY:
        raise ValueError(
            f"Nieznany zestaw parametrów: {zestaw}. "
            f"Dostępne: {', '.join(PARAMETRY)}"
        )

    parametry = PARAMETRY[zestaw]
    if seed is None:
        seed = token_bytes(32)
    if len(seed) != 32:
        raise ValueError("Seed generatora kluczy musi mieć 32 bajty.")

    rozszerzenie = sha3_512(seed).digest()
    rho = rozszerzenie[:32]
    sigma = rozszerzenie[32:]

    A = _generuj_macierz(rho, parametry.k)
    s = _wektor_cbd(sigma, 0, parametry.k, parametry.eta1)
    e = _wektor_cbd(sigma, parametry.k, parametry.k, parametry.eta1)

    As = Rq.macierz_razy_wektor(A, s)
    t = [Rq.dodaj(As[i], e[i]) for i in range(parametry.k)]

    return (rho, t), s


def EncK_PKE(
    klucz_publiczny: KluczPubliczny,
    wiadomosc: bytes,
    zestaw: str = "ML-KEM-768",
    losowosc: Optional[bytes] = None,
) -> Szyfrogram:
    """Szyfruje 32 bajty: u = A^T*r + e1, v = t^T*r + e2 + mu."""
    if zestaw not in PARAMETRY:
        raise ValueError(f"Nieznany zestaw parametrów: {zestaw}.")

    parametry = PARAMETRY[zestaw]
    rho, t = klucz_publiczny

    if len(rho) != 32:
        raise ValueError("rho w kluczu publicznym musi mieć 32 bajty.")
    if len(t) != parametry.k:
        raise ValueError("Klucz publiczny nie pasuje do zestawu parametrów.")

    if losowosc is None:
        losowosc = token_bytes(32)
    if len(losowosc) != 32:
        raise ValueError("Losowość szyfrowania musi mieć 32 bajty.")

    A = _generuj_macierz(rho, parametry.k)
    mu = _wiadomosc_na_wielomian(wiadomosc)

    r = _wektor_cbd(losowosc, 0, parametry.k, parametry.eta1)
    e1 = _wektor_cbd(losowosc, parametry.k, parametry.k, parametry.eta2)
    e2 = _prf_cbd(losowosc, 2 * parametry.k, parametry.eta2)

    Atr = Rq.transpozycja_razy_wektor(A, r)
    u = [Rq.dodaj(Atr[i], e1[i]) for i in range(parametry.k)]

    tr = Rq.iloczyn_skalarny(t, r)
    v = Rq.dodaj(Rq.dodaj(tr, e2), mu)

    return u, v


def DecK_PKE(
    klucz_prywatny: KluczPrywatny,
    szyfrogram: Szyfrogram,
    zestaw: str = "ML-KEM-768",
) -> bytes:
    """Deszyfruje przez mu' = v - s^T*u i dekoduje 256 bitów."""
    if zestaw not in PARAMETRY:
        raise ValueError(f"Nieznany zestaw parametrów: {zestaw}.")

    parametry = PARAMETRY[zestaw]
    u, v = szyfrogram

    if len(klucz_prywatny) != parametry.k:
        raise ValueError("Klucz prywatny nie pasuje do parametrów.")
    if len(u) != parametry.k:
        raise ValueError("Szyfrogram nie pasuje do parametrów.")
    if len(v) != N:
        raise ValueError(f"Wielomian v musi mieć {N} współczynników.")

    su = Rq.iloczyn_skalarny(klucz_prywatny, u)
    odzyskany_wielomian = Rq.odejmij(v, su)
    return _wielomian_na_wiadomosc(odzyskany_wielomian)


# ============================================================
# Przykład uruchomienia
# ============================================================

if __name__ == "__main__":
    zestaw = "ML-KEM-768"
    wiadomosc = b"To jest wiadomosc 32-bajtowa!!!!"
    assert len(wiadomosc) == 32

    pk, sk = KeyGenK_PKE(zestaw)
    ct = EncK_PKE(pk, wiadomosc, zestaw)
    odzyskana = DecK_PKE(sk, ct, zestaw)

    print("Zestaw parametrów:", zestaw)
    print("Wiadomość:        ", wiadomosc)
    print("Odzyskana:        ", odzyskana)
    print("Deszyfrowanie OK: ", odzyskana == wiadomosc)


Zestaw parametrów: ML-KEM-768
Wiadomość:         b'To jest wiadomosc 32-bajtowa!!!!'
Odzyskana:         b'To jest wiadomosc 32-bajtowa!!!!'
Deszyfrowanie OK:  True


## MQ PKE

In [8]:
# Uwaga: to jest demonstracja dydaktyczna, NIE bezpieczna kryptograficznie.

import numpy as np

q = 3329

# ===========================================
# Pomocnicze: mod q
# ===========================================
def mod(x, q=q):
    return np.mod(x, q)

# ===========================================
# Odwrotność macierzy mod q (Gauss-Jordan)
# ===========================================
def inv_mod_scalar(a, q=q):
    a %= q
    if a == 0:
        raise ValueError("0 nie ma odwrotności modulo q.")
    return pow(int(a), -1, int(q))

def inv_mod_matrix(A, q=q):
    A = np.asarray(A, dtype=np.int64) % q
    n, m = A.shape
    if n != m:
        raise ValueError("Macierz musi być kwadratowa.")
    I = np.eye(n, dtype=np.int64)
    aug = np.concatenate([A, I], axis=1) % q

    for col in range(n):
        pivot = None
        for r in range(col, n):
            if aug[r, col] % q != 0:
                pivot = r
                break
        if pivot is None:
            raise ValueError("Macierz nieodwracalna modulo q.")

        if pivot != col:
            aug[[col, pivot], :] = aug[[pivot, col], :]

        inv_p = inv_mod_scalar(aug[col, col], q)
        aug[col, :] = (aug[col, :] * inv_p) % q

        for r in range(n):
            if r == col:
                continue
            factor = aug[r, col] % q
            if factor != 0:
                aug[r, :] = (aug[r, :] - factor * aug[col, :]) % q

    return aug[:, n:] % q

def random_invertible_matrix(n, q=q, seed=None, max_tries=10000):
    rng = np.random.default_rng(seed)
    for _ in range(max_tries):
        M = rng.integers(0, q, size=(n, n), dtype=np.int64)
        try:
            _ = inv_mod_matrix(M, q)
            return M % q
        except ValueError:
            pass
    raise RuntimeError("Nie udało się wylosować odwracalnej macierzy.")

# ============================================
# Mapy afiniczne
# ============================================
def affine_apply(M, b, x, q=q):
    x = mod(np.asarray(x, dtype=np.int64), q)
    return mod(M @ x + b, q)

def affine_apply_inverse(M_inv, b, y, q=q):
    y = mod(np.asarray(y, dtype=np.int64), q)
    return mod(M_inv @ (y - b), q)

# ============================================
# Centralna mapa P (kwadratowa)
# ============================================
def P(u, q=q):
    """
    u = (u1,u2,u3) -> v = (v1,v2,v3)
    v1 = u1 + u2*u3
    v2 = u2 + u3
    v3 = u3
    """
    u = mod(np.asarray(u, dtype=np.int64), q)
    u1, u2, u3 = u
    v1 = (u1 + u2 * u3) % q
    v2 = (u2 + u3) % q
    v3 = u3 % q
    return np.array([v1, v2, v3], dtype=np.int64)

def P_inv(v, q=q):
    """
    Odwrócenie P:
      u3 = v3
      u2 = v2 - u3
      u1 = v1 - u2*u3
    """
    v = mod(np.asarray(v, dtype=np.int64), q)
    v1, v2, v3 = v
    u3 = v3 % q
    u2 = (v2 - u3) % q
    u1 = (v1 - (u2 * u3)) % q
    return np.array([u1, u2, u3], dtype=np.int64)

# ====================================================
# MQ.PKE
# ====================================================
def KeyGenMQ(q=q, seed=42):
    rng = np.random.default_rng(seed)

    # T: x -> u = A x + a
    A = random_invertible_matrix(3, q=q, seed=seed+1)
    a = rng.integers(0, q, size=(3,), dtype=np.int64) % q
    A_inv = inv_mod_matrix(A, q)

    # S: v -> y = B v + b
    B = random_invertible_matrix(3, q=q, seed=seed+2)
    b = rng.integers(0, q, size=(3,), dtype=np.int64) % q
    B_inv = inv_mod_matrix(B, q)

    pk = {"q": q, "A": A, "a": a, "B": B, "b": b}   # dla demo trzymamy też A,B; w prawdziwym PKE pk to "F"
    sk = {"q": q, "A": A, "a": a, "A_inv": A_inv,
          "B": B, "b": b, "B_inv": B_inv}
    return pk, sk

def EncMQ(pk, x):
    """
    Szyfrowanie: c = F(x) = S(P(T(x)))
      u = A x + a
      v = P(u)
      c = B v + b
    """
    q = pk["q"]
    x = mod(np.asarray(x, dtype=np.int64), q)
    u = affine_apply(pk["A"], pk["a"], x, q)
    v = P(u, q)
    c = affine_apply(pk["B"], pk["b"], v, q)
    return c

def DecMQ(sk, c):
    """
    Deszyfrowanie: x = T^{-1}(P^{-1}(S^{-1}(c)))
      v = B^{-1}(c - b)
      u = P^{-1}(v)
      x = A^{-1}(u - a)
    """
    q = sk["q"]
    c = mod(np.asarray(c, dtype=np.int64), q)
    v = affine_apply_inverse(sk["B_inv"], sk["b"], c, q)
    u = P_inv(v, q)
    x = affine_apply_inverse(sk["A_inv"], sk["a"], u, q)
    return x

# =============================================
# Demonstracja działania
# ===========================================
def demo():
    pk, sk = KeyGenMQ(q=q, seed=42)

    print("=== Parametry (tajne) ===")
    print("A=\n", pk["A"])
    print("a=", pk["a"])
    print("B=\n", pk["B"])
    print("b=", pk["b"])

    x = np.array([1, 4, 2], dtype=np.int64)
    c = EncMQ(pk, x)
    x2 = DecMQ(sk, c)

    print("\nWiadomość x:", x)
    print("Szyfrogram c:", c)
    print("Odszyfrowane x':", x2)
    print("Poprawność:", np.array_equal(x % q, x2 % q))

demo()



=== Parametry (tajne) ===
A=
 [[1682 2171 1335]
 [ 145 1921   66]
 [ 920 2793 1546]]
a= [ 297 2576 2179]
B=
 [[2219  408 2726]
 [ 859 2216 1350]
 [ 294 3226 1780]]
b= [1461 1441 2858]

Wiadomość x: [1 4 2]
Szyfrogram c: [ 818  830 1854]
Odszyfrowane x': [1 4 2]
Poprawność: True


## ClassicMcElliece PKE

In [9]:
# Uwaga: to jest demonstracja dydaktyczna, NIE bezpieczna kryptograficznie.

from __future__ import annotations

from dataclasses import dataclass
from hashlib import shake_256
from math import comb
import secrets
from typing import List, Sequence, Tuple

# ===========================================
# Parametry kryptosystemu
# ===========================================

M = 12
N = 3488
T = 64
R = M * T
Q = 1 << M
FIELD_MASK = Q - 1
FIELD_MODULUS = 0x1009  # x^12 + x^3 + 1
PLAINTEXT_BYTES = (comb(N, T).bit_length() - 1) // 8
CIPHERTEXT_BYTES = R // 8
PUBLIC_KEY_BYTES = R * (N - R) // 8

# ============================================
# Arytmetyka w ciele GF(2^12)
# ============================================


def gf_mul(a: int, b: int) -> int:
    """Multiplication in GF(2^12), polynomial basis, little-endian bits."""
    z = 0
    for _ in range(M):
        if b & 1:
            z ^= a
        b >>= 1
        a <<= 1
        if a & Q:
            a ^= FIELD_MODULUS
    return z & FIELD_MASK


def gf_square(a: int) -> int:
    return gf_mul(a, a)


def gf_pow(a: int, exponent: int) -> int:
    z = 1
    while exponent:
        if exponent & 1:
            z = gf_mul(z, a)
        a = gf_square(a)
        exponent >>= 1
    return z


def gf_inv(a: int) -> int:
    if a == 0:
        raise ZeroDivisionError("inverse of zero")
    return gf_pow(a, Q - 2)

#===================================================================
# Arytmetyka w ciele wielomianów
# ==================================================================

def p_trim(a: List[int]) -> List[int]:
    while len(a) > 1 and a[-1] == 0:
        a.pop()
    return a


def p_add(a: Sequence[int], b: Sequence[int]) -> List[int]:
    n = max(len(a), len(b))
    z = [0] * n
    for i in range(n):
        z[i] = (a[i] if i < len(a) else 0) ^ (b[i] if i < len(b) else 0)
    return p_trim(z)


def p_mul(a: Sequence[int], b: Sequence[int]) -> List[int]:
    z = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        if x:
            for j, y in enumerate(b):
                if y:
                    z[i + j] ^= gf_mul(x, y)
    return p_trim(z)


def p_divmod(a: Sequence[int], b: Sequence[int]) -> Tuple[List[int], List[int]]:
    r = p_trim(list(a))
    d = p_trim(list(b))
    if d == [0]:
        raise ZeroDivisionError("polynomial division by zero")
    q = [0] * max(1, len(r) - len(d) + 1)
    inv_lead = gf_inv(d[-1])
    while r != [0] and len(r) >= len(d):
        k = len(r) - len(d)
        c = gf_mul(r[-1], inv_lead)
        q[k] ^= c
        for j, x in enumerate(d):
            r[j + k] ^= gf_mul(c, x)
        p_trim(r)
    return p_trim(q), p_trim(r)


def p_mod(a: Sequence[int], modulus: Sequence[int]) -> List[int]:
    return p_divmod(a, modulus)[1]


def p_mulmod(a: Sequence[int], b: Sequence[int], modulus: Sequence[int]) -> List[int]:
    return p_mod(p_mul(a, b), modulus)


def p_squaremod(a: Sequence[int], modulus: Sequence[int]) -> List[int]:
    return p_mulmod(a, a, modulus)


def p_eval(a: Sequence[int], x: int) -> int:
    z = 0
    for c in reversed(a):
        z = gf_mul(z, x) ^ c
    return z


def p_gcd(a: Sequence[int], b: Sequence[int]) -> List[int]:
    x, y = p_trim(list(a)), p_trim(list(b))
    while y != [0]:
        x, y = y, p_mod(x, y)
    if x == [0]:
        return [0]
    inv = gf_inv(x[-1])
    return [gf_mul(c, inv) for c in x]


def p_invmod(a: Sequence[int], modulus: Sequence[int]) -> List[int]:
    r0, r1 = list(modulus), p_mod(a, modulus)
    s0, s1 = [0], [1]
    while r1 != [0]:
        q, r2 = p_divmod(r0, r1)
        r0, r1 = r1, r2
        s0, s1 = s1, p_add(s0, p_mul(q, s1))
    if len(r0) != 1 or r0[0] == 0:
        raise ZeroDivisionError("polynomial is not invertible")
    k = gf_inv(r0[0])
    return p_mod([gf_mul(c, k) for c in s0], modulus)


def p_sqrtmod(a: Sequence[int], modulus: Sequence[int]) -> List[int]:
    # In GF((2^m)^t), square root is a^(2^(m*t-1)).
    z = p_mod(a, modulus)
    for _ in range(M * T - 1):
        z = p_squaremod(z, modulus)
    return z

# =====================================================
# Kod Goppa
# =====================================================

def _random_monic_polynomial(degree: int) -> List[int]:
    return [secrets.randbelow(Q) for _ in range(degree)] + [1]


def _is_irreducible_degree_64(f: Sequence[int]) -> bool:
    # Rabin test. Since 64 has only prime divisor 2, test gcd(x^(q^32)-x,f)=1
    # and x^(q^64)-x=0 mod f.
    x = [0, 1]
    h = x
    checkpoint = None
    for i in range(1, T + 1):
        for _ in range(M):
            h = p_squaremod(h, f)
        if i == T // 2:
            checkpoint = h[:]
    return p_gcd(p_add(checkpoint, x), f) == [1] and p_trim(p_add(h, x)) == [0]


def _generate_goppa_polynomial() -> List[int]:
    while True:
        g = _random_monic_polynomial(T)
        if _is_irreducible_degree_64(g):
            return g


def _column_for(alpha: int, g: Sequence[int]) -> int:
    """One mt-bit column of the canonical alternant parity-check matrix."""
    inv = gf_inv(p_eval(g, alpha))
    power = 1
    col = 0
    for j in range(T):
        v = gf_mul(power, inv)
        col |= v << (j * M)
        power = gf_mul(power, alpha)
    return col


def _systematize_first_r(columns: Sequence[int]):
    """Return rows of [I|T] and original left columns; fail if left block singular."""
    # Convert H to mt row integers. Bit i in a row is column i.
    rows = [0] * R
    for i, col in enumerate(columns):
        c = col
        while c:
            lsb = c & -c
            bit = lsb.bit_length() - 1
            rows[bit] |= 1 << i
            c ^= lsb

    # RREF using exactly columns 0..R-1, applying row operations to all columns.
    for pivot in range(R):
        k = next((k for k in range(pivot, R) if (rows[k] >> pivot) & 1), None)
        if k is None:
            return None
        rows[pivot], rows[k] = rows[k], rows[pivot]
        pr = rows[pivot]
        for k in range(R):
            if k != pivot and ((rows[k] >> pivot) & 1):
                rows[k] ^= pr

    mask = (1 << (N - R)) - 1
    t_rows = [((row >> R) & mask) for row in rows]
    return t_rows, list(columns[:R])

# ==========================================
# Niederreiter PKE
# ==========================================

@dataclass(frozen=True)
class PublicKey:
    """Rows of T in the systematic matrix H=[I|T]."""
    t_rows: Tuple[int, ...]

    def to_bytes(self) -> bytes:
        width = (N - R) // 8
        return b"".join(x.to_bytes(width, "little") for x in self.t_rows)

    @classmethod
    def from_bytes(cls, data: bytes) -> "PublicKey":
        if len(data) != PUBLIC_KEY_BYTES:
            raise ValueError(f"public key must have {PUBLIC_KEY_BYTES} bytes")
        width = (N - R) // 8
        return cls(tuple(int.from_bytes(data[i:i+width], "little")
                         for i in range(0, len(data), width)))


@dataclass(frozen=True)
class SecretKey:
    g: Tuple[int, ...]
    support: Tuple[int, ...]
    left_columns: Tuple[int, ...]


def KeyGenMcEliece() -> Tuple[PublicKey, SecretKey]:
    """Generate a mceliece348864 Niederreiter-PKE key pair.

    The non-'f' behavior is modeled by retrying the field ordering until the
    first mt columns are nonsingular, rather than moving pivot columns.
    """
    g = _generate_goppa_polynomial()
    base = list(range(Q))
    while True:
        # A uniformly random ordering, truncated to n positions.
        support = base[:]
        secrets.SystemRandom().shuffle(support)
        support = support[:N]
        columns = [_column_for(a, g) for a in support]
        reduced = _systematize_first_r(columns)
        if reduced is not None:
            t_rows, left_columns = reduced
            return (PublicKey(tuple(t_rows)),
                    SecretKey(tuple(g), tuple(support), tuple(left_columns)))


def _unrank_weight_t(rank: int) -> int:
    if rank < 0 or rank >= comb(N, T):
        raise ValueError("plaintext integer is outside the fixed-weight domain")
    e = 0
    x = N - 1
    for w in range(T, 0, -1):
        while comb(x, w) > rank:
            x -= 1
        e |= 1 << x
        rank -= comb(x, w)
        x -= 1
    return e


def _rank_weight_t(e: int) -> int:
    positions = [i for i in range(N) if (e >> i) & 1]
    if len(positions) != T:
        raise ValueError("decoded vector does not have weight t")
    return sum(comb(pos, j + 1) for j, pos in enumerate(positions))


def EncMcEliece(public_key: PublicKey, plaintext: bytes) -> bytes:
    """Encrypt at most PLAINTEXT_BYTES bytes; output a 96-byte syndrome.

    The plaintext length is prefixed in the ranked integer, so leading zero
    bytes round-trip. The resulting integer must be below C(n,t).
    """
    if len(plaintext) > PLAINTEXT_BYTES - 1:
        raise ValueError(f"plaintext is limited to {PLAINTEXT_BYTES - 1} bytes")
    packed = bytes([len(plaintext)]) + plaintext
    rank = int.from_bytes(packed, "big")
    e = _unrank_weight_t(rank)
    left = e & ((1 << R) - 1)
    right = e >> R
    syndrome = left
    x = right
    # H e^T = e_left + T e_right.
    for row_index, row in enumerate(public_key.t_rows):
        if (row & x).bit_count() & 1:
            syndrome ^= 1 << row_index
    return syndrome.to_bytes(CIPHERTEXT_BYTES, "little")


def _patterson_decode(g: Sequence[int], support: Sequence[int], syndrome_fields: Sequence[int]) -> int:
    s = p_trim(list(syndrome_fields))
    if s == [0]:
        raise ValueError("zero syndrome does not encode a weight-t error")
    inv_s = p_invmod(s, g)
    tau = p_sqrtmod(p_add(inv_s, [0, 1]), g)

    # Extended Euclid until remainder degree <= t/2. Track coefficient of tau.
    r0, r1 = list(g), tau
    b0, b1 = [0], [1]
    while len(r1) - 1 > T // 2:
        q, r2 = p_divmod(r0, r1)
        r0, r1 = r1, r2
        b0, b1 = b1, p_add(b0, p_mul(q, b1))
    a, b = r1, b1

    a2 = p_mul(a, a)
    b2 = p_mul(b, b)
    sigma = p_add(a2, [0] + b2)
    if sigma == [0]:
        raise ValueError("Patterson decoder produced the zero locator")

    errors = 0
    for i, alpha in enumerate(support):
        if p_eval(sigma, alpha) == 0:
            errors |= 1 << i
    if errors.bit_count() != T:
        raise ValueError("decoding failure: locator does not have t support roots")
    return errors


def DecMcEliece(secret_key: SecretKey, ciphertext: bytes) -> bytes:
    """Decrypt a 96-byte syndrome and recover the original byte string."""
    if len(ciphertext) != CIPHERTEXT_BYTES:
        raise ValueError(f"ciphertext must have {CIPHERTEXT_BYTES} bytes")
    c = int.from_bytes(ciphertext, "little")
    if c >> R:
        raise ValueError("non-canonical ciphertext")

    # H_sys = A^{-1} H_orig, hence s_orig = A * s_sys.
    original = 0
    x = c
    while x:
        lsb = x & -x
        j = lsb.bit_length() - 1
        original ^= secret_key.left_columns[j]
        x ^= lsb
    s_fields = [(original >> (j * M)) & FIELD_MASK for j in range(T)]
    error = _patterson_decode(secret_key.g, secret_key.support, s_fields)

    # Essential validation against malformed ciphertexts.
    columns = [_column_for(a, secret_key.g) for a in secret_key.support]
    check_original = 0
    x = error
    while x:
        lsb = x & -x
        i = lsb.bit_length() - 1
        check_original ^= columns[i]
        x ^= lsb
    if check_original != original:
        raise ValueError("decoding failure: syndrome verification failed")

    rank = _rank_weight_t(error)
    width = max(1, (rank.bit_length() + 7) // 8)
    packed = rank.to_bytes(width, "big")
    length = packed[0]
    if length != len(packed) - 1 or length > PLAINTEXT_BYTES - 1:
        raise ValueError("decoded fixed-weight vector is not a valid plaintext encoding")
    return packed[1:]

# =====================================
# Testy i demonstracja działania
# =====================================

def demo(message: bytes = b"Classic McEliece demo") -> bool:
    """Run a complete key generation, encryption and decryption demonstration.

    Returns True when the recovered message is identical to the input. Because
    this is a pure-Python reference implementation with production-size
    mceliece348864 parameters, key generation can take a long time.
    """
    if not isinstance(message, bytes):
        raise TypeError("message must be bytes")
    if len(message) > PLAINTEXT_BYTES - 1:
        raise ValueError(f"demo message is limited to {PLAINTEXT_BYTES - 1} bytes")

    print("=" * 72)
    print("Classic McEliece / Niederreiter PKE demonstration: mceliece348864")
    print("=" * 72)
    print(f"Parameters: m={M}, n={N}, t={T}, mt={R}")
    print(f"Maximum plaintext length: {PLAINTEXT_BYTES - 1} bytes")
    print(f"Ciphertext length:        {CIPHERTEXT_BYTES} bytes")
    print(f"Public-key length:        {PUBLIC_KEY_BYTES} bytes")
    print(f"Plaintext:                {message!r}")

    print("\n[1/3] Generating the Goppa-code key pair...")
    start = time.perf_counter()
    public_key, secret_key = KeyGenMcEliece()
    keygen_time = time.perf_counter() - start
    serialized_public_key = public_key.to_bytes()
    print(f"      done in {keygen_time:.3f} s")
    print(f"      serialized public key: {len(serialized_public_key)} bytes")

    print("\n[2/3] Encrypting the message...")
    start = time.perf_counter()
    ciphertext = EncMcEliece(public_key, message)
    encryption_time = time.perf_counter() - start
    print(f"      done in {encryption_time:.6f} s")
    print(f"      ciphertext ({len(ciphertext)} bytes): {ciphertext.hex()}")

    print("\n[3/3] Decrypting with Patterson decoding...")
    start = time.perf_counter()
    recovered = DecMcEliece(secret_key, ciphertext)
    decryption_time = time.perf_counter() - start
    print(f"      done in {decryption_time:.6f} s")
    print(f"      recovered plaintext: {recovered!r}")

    success = secrets.compare_digest(message, recovered)
    print("\nResult:", "SUCCESS" if success else "FAILURE")
    print("The recovered plaintext is identical to the original." if success
          else "The recovered plaintext differs from the original.")
    print("=" * 72)
    return success


## HQC.PKE

In [10]:
# Uwaga: to jest demonstracja dydaktyczna, NIE bezpieczna kryptograficznie.

from dataclasses import dataclass
import secrets
from typing import Tuple

# ===================================================
# Parametry HQC-128 (submission 2023)
# ===================================================

N, N1, N2, K = 17669, 46, 384, 16
DELTA = 15
OMEGA, OMEGA_R, OMEGA_E = 66, 75, 75
MASK_N = (1 << N) - 1

# ===================================================
# GF(2^8), wielomian x^8+x^4+x^3+x^2+1 = 0x11d
# ===================================================

GF_POLY = 0x11D
GF_EXP = [0] * 510
GF_LOG = [0] * 256
x = 1
for i in range(255):
    GF_EXP[i] = x
    GF_LOG[x] = i
    x <<= 1
    if x & 0x100:
        x ^= GF_POLY
for i in range(255, 510):
    GF_EXP[i] = GF_EXP[i - 255]

def gf_mul(a: int, b: int) -> int:
    return 0 if not a or not b else GF_EXP[GF_LOG[a] + GF_LOG[b]]

def gf_inv(a: int) -> int:
    if not a: raise ZeroDivisionError
    return GF_EXP[255 - GF_LOG[a]]

def gf_pow_alpha(e: int) -> int:
    return GF_EXP[e % 255]

def poly_eval_low(p, z):
    r = 0
    for a in reversed(p): r = gf_mul(r, z) ^ a
    return r

def gf_solve(A, b):
    """Eliminacja Gaussa nad GF(256)."""
    n = len(b)
    M = [list(A[i]) + [b[i]] for i in range(n)]
    for col in range(n):
        piv = next((r for r in range(col, n) if M[r][col]), None)
        if piv is None: raise ValueError("osobliwy układ RS")
        M[col], M[piv] = M[piv], M[col]
        inv = gf_inv(M[col][col])
        M[col] = [gf_mul(v, inv) for v in M[col]]
        for r in range(n):
            if r != col and M[r][col]:
                f = M[r][col]
                M[r] = [u ^ gf_mul(f, v) for u, v in zip(M[r], M[col])]
    return [M[i][-1] for i in range(n)]

# =====================================================================
# Kod Reed-Solomon [46,16,31], korekcja do 15 symboli
# Reprezentacja słowa jako współczynniki c_0,...,c_45.
# Warunki kodu: c(alpha^j)=0 dla j=1,...,30.
# =====================================================================

_RS_A = [[gf_pow_alpha((j + 1) * i) for i in range(30)] for j in range(30)]

def rs_encode(msg: bytes) -> bytes:
    if len(msg) != K: raise ValueError("HQC-128 szyfruje dokładnie 16 bajtów")
    # Systematycznie: wiadomość zajmuje współczynniki 30..45.
    rhs = []
    for j in range(1, 31):
        s = 0
        for t, m in enumerate(msg): s ^= gf_mul(m, gf_pow_alpha(j * (30 + t)))
        rhs.append(s)
    parity = gf_solve(_RS_A, rhs)
    return bytes(parity) + msg

def rs_syndromes(c: bytes):
    return [sum_gf(gf_mul(v, gf_pow_alpha(j * i)) for i, v in enumerate(c))
            for j in range(1, 31)]

def sum_gf(values):
    r = 0
    for v in values: r ^= v
    return r

def berlekamp_massey(S):
    C = [1] + [0] * len(S); B = [1] + [0] * len(S)
    L, m, b = 0, 1, 1
    for n in range(len(S)):
        d = S[n]
        for i in range(1, L + 1): d ^= gf_mul(C[i], S[n - i])
        if d == 0:
            m += 1; continue
        T = C[:]; coef = gf_mul(d, gf_inv(b))
        for i in range(len(S) + 1 - m):
            if B[i]: C[i + m] ^= gf_mul(coef, B[i])
        if 2 * L <= n:
            L, B, b, m = n + 1 - L, T, d, 1
        else:
            m += 1
    return C[:L + 1], L

def rs_decode(received: bytes) -> bytes:
    if len(received) != N1: raise ValueError("niepoprawna długość słowa RS")
    c = bytearray(received); S = rs_syndromes(c)
    if not any(S): return bytes(c[30:46])
    locator, L = berlekamp_massey(S)
    if L > DELTA: raise ValueError("RS: zbyt wiele błędów")
    pos = [i for i in range(N1) if poly_eval_low(locator, gf_pow_alpha(-i)) == 0]
    if len(pos) != L: raise ValueError("RS: nie udało się zlokalizować błędów")
    A = [[gf_pow_alpha((j + 1) * i) for i in pos] for j in range(L)]
    mags = gf_solve(A, S[:L])
    for i, e in zip(pos, mags): c[i] ^= e
    if any(rs_syndromes(c)): raise ValueError("RS: korekcja nie powiodła się")
    return bytes(c[30:46])

# ================================================================
# Kod Reed-Muller RM(1,7), [384,8]
# ================================================================

def rm_encode_symbol(a: int) -> int:
    block = 0
    for point in range(128):
        bit = a & 1
        for j in range(7): bit ^= ((a >> (j + 1)) & 1) & ((point >> j) & 1)
        block |= bit << point
    return block | (block << 128) | (block << 256)

def fwht(v):
    h = 1
    while h < len(v):
        for i in range(0, len(v), 2*h):
            for j in range(i, i+h):
                a, b = v[j], v[j+h]; v[j], v[j+h] = a+b, a-b
        h *= 2

def rm_decode_symbol(word: int) -> int:
    # Głosowanie trzech kopii, potem dekodowanie maksimum korelacji RM(1,7).
    b0 = word & ((1 << 128) - 1)
    b1 = (word >> 128) & ((1 << 128) - 1)
    b2 = (word >> 256) & ((1 << 128) - 1)
    vals = [1 if (((b0>>i)&1)+((b1>>i)&1)+((b2>>i)&1)) < 2 else -1 for i in range(128)]
    fwht(vals)
    idx = max(range(128), key=lambda i: abs(vals[i]))
    constant = 0 if vals[idx] >= 0 else 1
    return constant | (idx << 1)

def code_encode(msg: bytes) -> int:
    rs = rs_encode(msg); out = 0
    for i, a in enumerate(rs): out |= rm_encode_symbol(a) << (N2 * i)
    return out  # 46*384=17664 bitów; pięć końcowych współczynników to zera

def code_decode(word: int) -> bytes:
    symbols = bytes(rm_decode_symbol((word >> (N2*i)) & ((1<<N2)-1)) for i in range(N1))
    return rs_decode(symbols)

# =================================================
# Arytmetyka w pierścieniu wielomianów
# R = GF(2)[X]/(X^n-1)
# =================================================

def rot_n(a: int, k: int) -> int:
    k %= N
    return ((a << k) | (a >> (N-k))) & MASK_N

def mul_sparse(a: int, positions: Tuple[int, ...]) -> int:
    r = 0
    for p in positions: r ^= rot_n(a, p)
    return r

def sample_fixed_weight(weight: int) -> Tuple[int, ...]:
    return tuple(sorted(secrets.SystemRandom().sample(range(N), weight)))

def sparse_to_int(p):
    r = 0
    for i in p: r |= 1 << i
    return r

def random_ring_element():
    return int.from_bytes(secrets.token_bytes((N+7)//8), 'little') & MASK_N

#==================================================
# HQC.PKE
#==================================================

@dataclass(frozen=True)
class KluczPubliczny:
    h: int
    s: int

@dataclass(frozen=True)
class KluczPrywatny:
    x: Tuple[int, ...]
    y: Tuple[int, ...]

@dataclass(frozen=True)
class Szyfrogram:
    u: int
    v: int


def KeyGenHQC():
    x = sample_fixed_weight(OMEGA)
    y = sample_fixed_weight(OMEGA)
    h = random_ring_element()
    s = sparse_to_int(x) ^ mul_sparse(h, y)
    return KluczPubliczny(h, s), KluczPrywatny(x, y)

def EncHQC(pk: KluczPubliczny, wiadomosc: bytes) -> Szyfrogram:
    if len(wiadomosc) != K: raise ValueError("wiadomość musi mieć 16 bajtów")
    r1 = sample_fixed_weight(OMEGA_R)
    r2 = sample_fixed_weight(OMEGA_R)
    e = sample_fixed_weight(OMEGA_E)
    u = sparse_to_int(r1) ^ mul_sparse(pk.h, r2)
    v = code_encode(wiadomosc) ^ mul_sparse(pk.s, r2) ^ sparse_to_int(e)
    return Szyfrogram(u, v)

def DecHQC(sk: KluczPrywatny, ct: Szyfrogram) -> bytes:
    # v + u*y = Encode(m) + x*r2 + r1*y + e w charakterystyce 2.
    return code_decode(ct.v ^ mul_sparse(ct.u, sk.y))

#======================================================
# Demonstracja działania i testy
#======================================================

def _testy():
    # Najpierw test pełnej przestrzeni symboli RM.
    for a in range(256):
        assert rm_decode_symbol(rm_encode_symbol(a)) == a
    # Test RS z 15 błędnymi symbolami.
    for _ in range(3):
        m = secrets.token_bytes(16); c = bytearray(rs_encode(m))
        for p in secrets.SystemRandom().sample(range(46), 15): c[p] ^= secrets.randbelow(255)+1
        assert rs_decode(bytes(c)) == m
    # Test całego HQC.PKE.
    for _ in range(5):
        pk, sk = KeyGenHQC(); m = secrets.token_bytes(16)
        ct = EncHQC(pk, m); got = DecHQC(sk, ct)
        assert got == m, (m.hex(), got.hex())
    print("[OK] RM(1,7): 256/256")
    print("[OK] RS[46,16,31]: 3/3 z 15 błędami symbolowymi")
    print("[OK] HQC-128 PKE: 5/5")

if __name__ == "__main__":
    _testy()

[OK] RM(1,7): 256/256
[OK] RS[46,16,31]: 3/3 z 15 błędami symbolowymi
[OK] HQC-128 PKE: 5/5


## Transformacje Fujisaki-Okamoto

Pseudokod poszczególnych transformacji znajduje się w dołączonym pliku pdf. Można wykorzystać własne implementacje wykonane na podstawie pseudokodu lub dostosować poniższe implementacje.

In [11]:
"""
Ogólne implementacje transformacji Fujisaki-Okamoto dla niekoniecznie bezbłędnych schematów PKE.

Założony interfejs bazowego PKE:
    pk, sk = KeyGen()
    c = Enc(pk, m, coins)
    m = Dec(sk, c)

Transformacje nie zakładają żadnej struktury kluczy ani wiadomości.
Wszelkie dodatkowe informacje wymagane przez transformację są
przechowywane w rozszerzonym kluczu tajnym zgodnie z pseudokodami.
"""

import os
import hashlib

BOTTOM = None


def random_message(length=32):
    return os.urandom(length)


# Przykładowe funkcje kryptograficzne (można podmienić)
def H(*args):
    return hashlib.sha256(repr(args).encode()).digest()


def G(*args):
    return hashlib.sha256(repr(args).encode()).digest()


def J(*args):
    return hashlib.sha256(repr(args).encode()).digest()


# --------------------------------------------------
# T(PKE,G) = PKE1
# --------------------------------------------------

def transform_T(KeyGen, Enc, Dec, G):

    def KeyGen1():
        pk, sk = KeyGen()
        return pk, (sk, pk)

    def Enc1(pk, m):
        r = G(m)
        return Enc(pk, m, r)

    def Dec1(sk1, c):
        sk, pk = sk1

        m = Dec(sk, c)
        if m is BOTTOM:
            return BOTTOM

        if Enc(pk, m, G(m)) != c:
            return BOTTOM

        return m

    return KeyGen1, Enc1, Dec1


# --------------------------------------------------
# U_{!=⊥}(PKE1,H)
# --------------------------------------------------

def transform_U_not_perp(KeyGenPKE1, EncPKE1, DecPKE1, H):

    def KeyGenKEM():
        pk1, sk1 = KeyGenPKE1()
        s = random_message()
        return pk1, (sk1, s)

    def Encaps(pk):
        m = random_message()
        c = EncPKE1(pk, m)
        K = H(m, c)
        return K, c

    def Decaps(sk, c):
        sk1, s = sk

        m = DecPKE1(sk1, c)

        if m is not BOTTOM:
            return H(m, c)

        return H(s, c)

    return KeyGenKEM, Encaps, Decaps


# --------------------------------------------------
# U_{!=⊥,m}(PKE1,H)
# --------------------------------------------------

def transform_U_not_perp_m(KeyGenPKE1, EncPKE1, DecPKE1, H):

    def KeyGenKEM():
        pk1, sk1 = KeyGenPKE1()
        s = random_message()
        return pk1, (sk1, s)

    def Encaps(pk):
        m = random_message()
        c = EncPKE1(pk, m)
        K = H(m)
        return K, c

    def Decaps(sk, c):
        sk1, s = sk

        m = DecPKE1(sk1, c)

        if m is not BOTTOM:
            return H(m)

        return H(s, c)

    return KeyGenKEM, Encaps, Decaps


# --------------------------------------------------
# U_{⊥}(PKE1,H)
# --------------------------------------------------

def transform_U_perp(KeyGenPKE1, EncPKE1, DecPKE1, H):

    def KeyGenKEM():
        return KeyGenPKE1()

    def Encaps(pk):
        m = random_message()
        c = EncPKE1(pk, m)
        K = H(m, c)
        return K, c

    def Decaps(sk, c):
        m = DecPKE1(sk, c)

        if m is BOTTOM:
            return BOTTOM

        return H(m, c)

    return KeyGenKEM, Encaps, Decaps


# --------------------------------------------------
# U_{⊥,m}(PKE1,H)
# --------------------------------------------------

def transform_U_perp_m(KeyGenPKE1, EncPKE1, DecPKE1, H):

    def KeyGenKEM():
        return KeyGenPKE1()

    def Encaps(pk):
        m = random_message()
        c = EncPKE1(pk, m)
        K = H(m)
        return K, c

    def Decaps(sk, c):
        m = DecPKE1(sk, c)

        if m is BOTTOM:
            return BOTTOM

        return H(m)

    return KeyGenKEM, Encaps, Decaps


# --------------------------------------------------
# Wariant U'
# --------------------------------------------------

def transform_U_prime(KeyGenPKE1, EncPKE1, DecPKE1, H, G, J):

    def KeyGenKEM():
        pk, sk1 = KeyGenPKE1()
        hpk = H(pk)
        s = random_message()
        return pk, (sk1, pk, hpk, s)

    def Encaps(pk):
        m = random_message()
        K = G(m, H(pk))
        c = EncPKE1(pk, m)
        return K, c

    def Decaps(sk, c):
        sk1, pk, hpk, s = sk

        m = DecPKE1(sk1, c)

        if m is not BOTTOM:
            return G(m, hpk)

        return J(s, c)

    return KeyGenKEM, Encaps, Decaps


# --------------------------------------------------
# T'(PKE,G) = PKE1
# --------------------------------------------------

def transform_T_prime(KeyGen, Enc, Dec, G, salt_len=32):

    def KeyGen1():
        pk, sk = KeyGen()
        return pk, (sk, pk)

    def Enc1(pk, mu):
        salt = os.urandom(salt_len)
        r = G(mu, salt)
        c = Enc(pk, mu, r)
        return (c, salt)

    def Dec1(sk1, ciphertext):
        sk, pk = sk1
        c, salt = ciphertext

        mu = Dec(sk, c)

        if mu is BOTTOM:
            return BOTTOM

        if Enc(pk, mu, G(mu, salt)) != c:
            return BOTTOM

        return mu

    return KeyGen1, Enc1, Dec1

# Protokół Fujioki

In [12]:
from hashlib import sha3_256

# K-PKE korzysta z globalnych N i Q, nadpisanych przez komórki McEliece i HQC
N, Q = 256, 3329


def pke_kpke(zestaw):
    return (
        lambda: KeyGenK_PKE(zestaw),
        lambda pk, m, r: EncK_PKE(pk, m, zestaw, r),
        lambda sk, c: DecK_PKE(sk, c, zestaw),
    )


KEM1_Gen, KEM1_Encaps, KEM1_Decaps = transform_U_not_perp(
    *transform_T(*pke_kpke("ML-KEM-512"), G), H
)
KEM2_Gen, KEM2_Encaps, KEM2_Decaps = transform_U_prime(
    *transform_T(*pke_kpke("ML-KEM-768"), G), H, G, J
)


def KDF(sid, k1, k2, k3):
    return sha3_256(repr((sid, k1, k2, k3)).encode()).digest()


def _sid(I, R, spk_i, spk_r, epk_i, c1, c2, c3):
    return (I, R, spk_i, spk_r, epk_i, c1, c2, c3)


def inicjator_start(spk_r):
    """Initiator: KEM1.Gen oraz KEM2.Encaps na kluczu statycznym respondenta."""
    epk_i, esk_i = KEM1_Gen()
    k1, c1 = KEM2_Encaps(spk_r)
    return esk_i, epk_i, k1, c1


def responder(spk_i, ssk_r, epk_i, c1):
    """Responder: dwa encaps oraz odszyfrowanie k1 z c1."""
    k2, c2 = KEM2_Encaps(spk_i)
    k3, c3 = KEM1_Encaps(epk_i)
    k1 = KEM2_Decaps(ssk_r, c1)
    return k1, k2, k3, c2, c3


def inicjator_koniec(ssk_i, esk_i, spk_i, spk_r, epk_i, c1, c2, c3, k1):
    """Initiator: decaps k2, k3 i wspólny klucz sesyjny."""
    k2 = KEM2_Decaps(ssk_i, c2)
    k3 = KEM1_Decaps(esk_i, c3)
    sid = _sid("Alice", "Bob", spk_i, spk_r, epk_i, c1, c2, c3)
    return KDF(sid, k1, k2, k3)


def responder_klucz(spk_i, spk_r, epk_i, c1, c2, c3, k1, k2, k3):
    sid = _sid("Alice", "Bob", spk_i, spk_r, epk_i, c1, c2, c3)
    return KDF(sid, k1, k2, k3)

In [13]:
def demo_fujioka():
    print("=== Protokół Fujioki (KEM1=ML-KEM-512, KEM2=ML-KEM-768) ===")

    spk_i, ssk_i = KEM2_Gen()
    spk_r, ssk_r = KEM2_Gen()

    esk_i, epk_i, k1_i, c1 = inicjator_start(spk_r)
    print("Initiator -> Responder: epk_i, c1")

    k1_r, k2, k3, c2, c3 = responder(spk_i, ssk_r, epk_i, c1)
    print("Responder -> Initiator: c2, c3")

    SK_i = inicjator_koniec(ssk_i, esk_i, spk_i, spk_r, epk_i, c1, c2, c3, k1_i)
    SK_r = responder_klucz(spk_i, spk_r, epk_i, c1, c2, c3, k1_r, k2, k3)

    assert k1_i == k1_r, "k1 powinno się zgadzać po obu stronach"
    assert SK_i == SK_r, "klucz sesyjny musi być identyczny"
    print("k1 OK:", k1_i.hex()[:32], "...")
    print("SK :", SK_i.hex())

    # test: zmodyfikowany c2 -> inny (lub odrzucony) klucz u inicjatora
    u, v = c2
    c2_zly = (u, [x ^ 1 for x in v])
    k2_bad = KEM2_Decaps(ssk_i, c2_zly)
    k3_ok = KEM1_Decaps(esk_i, c3)
    sid = _sid("Alice", "Bob", spk_i, spk_r, epk_i, c1, c2_zly, c3)
    SK_bad = KDF(sid, k1_i, k2_bad, k3_ok)
    assert SK_bad != SK_i, "po manipulacji c2 klucz sesyjny musi się różnić"
    print("Test manipulacji c2: klucze różne — OK")
    print("=== Sukces ===")


demo_fujioka()

=== Protokół Fujioki (KEM1=ML-KEM-512, KEM2=ML-KEM-768) ===
Initiator -> Responder: epk_i, c1
Responder -> Initiator: c2, c3
k1 OK: a12cee7798f0359de46e8f038a6430a1 ...
SK : 6058acf87645ac14931f5e723036f6e05b550a4b9ae668d133786149cbfd25f8
Test manipulacji c2: klucze różne — OK
=== Sukces ===
